# 01 — Baselines Tabulares

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)
**Fase:** 1 — Baselines tabulares

## Objetivo

Antes de invertir en el Transformer, establecemos un **piso de performance** con modelos tabulares estándar:

- **ELO solo** — usa directamente la probabilidad esperada del cálculo de ELO. Sin ML. Sanity check absoluto.
- **Logistic Regression** — modelo lineal, interpretable, cota inferior para modelos paramétricos.
- **XGBoost** — workhorse del ML tabular, casi siempre fuerte.
- **LightGBM** — alternativa más rápida, comparación.

## Por qué esto importa académicamente

Toda contribución de NLP moderna se valida contra un baseline más simple. Decir _"mi Transformer logra X log-loss"_ es vacío si no comparamos contra _"XGBoost logra Y"_. La métrica relevante es **delta sobre baseline**.

## Por qué esto importa prácticamente

Si el Transformer no llega a tiempo o falla, **estos modelos ya predicen el Mundial 2026**. No nos quedamos sin nada.

## Métricas (no usamos accuracy como métrica principal)

- **Log-loss** (cross-entropy) — penaliza la sobreconfianza incorrecta.
- **Brier score** — MSE de probabilidades.
- **ECE** (Expected Calibration Error) — qué tan bien la probabilidad reportada matchea la frecuencia empírica.
- **Accuracy** — referencia secundaria.

Toda la lógica está implementada en módulos reutilizables en `src/`:
- `src/data/feature_engineering.py` — pipeline leakage-free
- `src/eval/metrics.py` — Brier, ECE, log-loss, reliability
- `src/models/baselines.py` — wrappers de modelos

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
# Asegurar deps
import importlib, subprocess
for pkg in ['xgboost', 'lightgbm']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call(['pip', 'install', '-q', pkg])

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)

# Importar módulos del proyecto
from data.feature_engineering import (
    build_feature_matrix, temporal_split, prepare_xy,
    get_feature_columns, LABEL_NAMES,
)
from eval.metrics import (
    multiclass_log_loss, multiclass_brier_score, expected_calibration_error,
    reliability_curve_data, evaluate_all, uniform_baseline_metrics,
)
from models.baselines import get_all_baselines

DATA_INTERIM = Path(PROJECT_ROOT) / 'data' / 'interim'
REPORTS = Path(PROJECT_ROOT) / 'reports'
FIGURES = REPORTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
CHECKPOINTS = Path(PROJECT_ROOT) / 'checkpoints' / 'baselines'
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
print('✓ Imports completados')

---
## 2. Cargar datos de Fase 0.5

Esperamos el parquet `international_matches_with_elo.parquet` producido por `00b_data_augmentation.ipynb`.

In [ ]:
input_path = DATA_INTERIM / 'international_matches_with_elo.parquet'
assert input_path.exists(), f'No se encuentra {input_path}. Correr primero notebook 00b.'

df_matches = pd.read_parquet(input_path)
df_matches['date'] = pd.to_datetime(df_matches['date'])
print(f'Partidos cargados: {len(df_matches):,}')
print(f'Rango temporal: {df_matches.date.min().date()} a {df_matches.date.max().date()}')
df_matches.head(3)

---
## 3. Feature engineering

Llamada al módulo `src/data/feature_engineering.py`. Computa features rolling con histórico per-team, sin leakage.

Filtramos a partidos `>= 2014-01-01` para entrenamiento. Los partidos previos siguen alimentando el histórico (no se pierde señal), pero no entran al training.

In [ ]:
%%time
feature_df = build_feature_matrix(df_matches, min_date='2014-01-01')
print(f'Feature matrix: {len(feature_df):,} filas × {len(feature_df.columns)} columnas')
feature_df.head(3)

In [ ]:
# NaN por columna (cold-start de equipos sin historial suficiente)
nan_rates = feature_df.isna().mean().sort_values(ascending=False)
nan_rates = nan_rates[nan_rates > 0]
if len(nan_rates) > 0:
    print('Features con NaN:')
    print(nan_rates.to_frame('nan_rate').round(4))
else:
    print('Sin NaN en features.')

In [ ]:
# Distribución del target (sanity check)
target_dist = feature_df.result.value_counts(normalize=True).sort_index()
target_dist.index = [LABEL_NAMES[i] for i in target_dist.index]
print('Distribución del target (post-2014):')
print(target_dist.round(4))
target_dist.plot(kind='bar', color=['steelblue', 'gray', 'firebrick'], title='Distribución resultado')
plt.ylabel('Frecuencia'); plt.show()

---
## 4. Temporal split

**Crítico:** split por fecha, NUNCA aleatorio.

In [ ]:
TRAIN_CUTOFF = '2023-01-01'
VAL_CUTOFF = '2024-01-01'

train_df, val_df, test_df = temporal_split(feature_df, TRAIN_CUTOFF, VAL_CUTOFF)
print(f'Train: {len(train_df):>5} partidos (< {TRAIN_CUTOFF})')
print(f'Val:   {len(val_df):>5} partidos ({TRAIN_CUTOFF} – {VAL_CUTOFF})')
print(f'Test:  {len(test_df):>5} partidos (>= {VAL_CUTOFF})')

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    dist = d.result.value_counts(normalize=True).sort_index()
    dist.index = [LABEL_NAMES[i] for i in dist.index]
    print(f'\n{name}:', dict(zip(dist.index, dist.round(3))))

In [ ]:
# Preparar X, y para cada split
X_train, y_train = prepare_xy(train_df)
X_val, y_val = prepare_xy(val_df)
X_test, y_test = prepare_xy(test_df)

# Alinear columnas one-hot entre splits
all_cols = sorted(set(X_train.columns) | set(X_val.columns) | set(X_test.columns))
for X in [X_train, X_val, X_test]:
    for c in all_cols:
        if c not in X.columns:
            X[c] = 0.0
X_train = X_train[all_cols]
X_val = X_val[all_cols]
X_test = X_test[all_cols]

print(f'Features finales: {len(all_cols)}')
print(f'X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')

---
## 5. Entrenamiento de los 4 baselines

In [ ]:
%%time
models = get_all_baselines()
trained_models = {}

for name, model in models.items():
    print(f'\n→ Entrenando {name}...')
    if name == 'elo_only':
        model.fit(X_train, y_train)
    else:
        model.fit(X_train, y_train, X_val, y_val)
    trained_models[name] = model
    print(f'  ✓ {name} entrenado')

---
## 6. Evaluación: val + test, todos los modelos

In [ ]:
def evaluate_on(model, X, y, split_name):
    probs = model.predict_proba(X)
    return {**evaluate_all(y, probs, name=model.name), 'split': split_name}

results = []
for split_name, y_split in [('val', y_val), ('test', y_test)]:
    u = uniform_baseline_metrics(y_split)
    u['split'] = split_name
    results.append(u)

for name, model in trained_models.items():
    results.append(evaluate_on(model, X_val, y_val, 'val'))
    results.append(evaluate_on(model, X_test, y_test, 'test'))

results_df = pd.DataFrame(results)[['model', 'split', 'log_loss', 'brier', 'ece', 'accuracy', 'n']]
results_df = results_df.sort_values(['split', 'log_loss']).reset_index(drop=True)
results_df.round(4)

In [ ]:
for split in ['val', 'test']:
    print(f'\n--- {split.upper()} ---')
    print(results_df[results_df.split == split][['model', 'log_loss', 'brier', 'ece', 'accuracy']].round(4).to_string(index=False))

# Persistir
results_df.to_csv(REPORTS / 'baseline_results.csv', index=False)
print(f'\n✓ Tabla guardada: {REPORTS / "baseline_results.csv"}')

---
## 7. Reliability diagrams

Comparamos **confianza promedio** vs **accuracy empírica** en cada bin. Diagonal = calibración perfecta.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, (name, model) in zip(axes.flat, trained_models.items()):
    probs = model.predict_proba(X_test)
    data = reliability_curve_data(y_test, probs, n_bins=10)
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Calibración perfecta')
    valid = ~np.isnan(data['bin_confidence'])
    ax.plot(data['bin_confidence'][valid], data['bin_accuracy'][valid], 'o-', color='steelblue', linewidth=2, markersize=8)
    
    ax2 = ax.twinx()
    ax2.bar(data['bin_centers'], data['bin_counts'], width=0.08, alpha=0.2, color='gray')
    ax2.set_ylabel('# preds por bin', color='gray')
    
    ece = expected_calibration_error(y_test, probs)
    ax.set_title(f'{name} — ECE: {ece:.4f}')
    ax.set_xlabel('Confianza (max prob)')
    ax.set_ylabel('Accuracy empírica')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(loc='upper left')

plt.suptitle('Reliability diagrams (TEST set)', y=1.00, fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES / 'reliability_diagrams_test.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. Feature importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, name in zip(axes, ['xgboost', 'lightgbm']):
    fi = trained_models[name].feature_importance().head(20)
    fi.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Top 20 features — {name}')
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES / 'feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

**Sanity check esperado:** `elo_diff`, `expected_home_win_prob`, y las features de forma reciente deberían estar arriba. Si features ruidosas como `home_rest_days` dominan, hay algo raro.

---
## 9. Análisis estratificado

Stratificamos los errores del mejor modelo en TEST por (a) cercanía del partido (|ELO diff|), (b) clase de torneo.

In [ ]:
# Tomar el mejor modelo por log-loss en val
val_results = results_df[results_df.split == 'val'].set_index('model')
best_model_name = val_results.log_loss.idxmin()
best_model = trained_models[best_model_name] if best_model_name in trained_models else None
print(f'Mejor modelo por log-loss val: {best_model_name}')

if best_model is not None:
    probs_test = best_model.predict_proba(X_test)
    preds_test = probs_test.argmax(axis=1)
    
    test_with_preds = test_df.copy()
    test_with_preds['pred'] = preds_test
    test_with_preds['correct'] = (test_with_preds.pred == test_with_preds.result).astype(int)
    test_with_preds['nll'] = -np.log(np.clip(probs_test[np.arange(len(y_test)), y_test], 1e-9, 1.0))
    test_with_preds['abs_elo_diff'] = test_with_preds.elo_diff.abs()
    test_with_preds['elo_diff_bin'] = pd.cut(
        test_with_preds.abs_elo_diff,
        bins=[0, 100, 200, 400, 2000],
        labels=['<100', '100-200', '200-400', '>400'],
    )
    
    by_elo = test_with_preds.groupby('elo_diff_bin').agg(
        n=('result', 'size'), accuracy=('correct', 'mean'), mean_nll=('nll', 'mean'),
    ).round(3)
    print('\n--- Performance por |ELO diff| ---')
    print(by_elo)
    
    by_tournament = test_with_preds.groupby('tournament_class').agg(
        n=('result', 'size'), accuracy=('correct', 'mean'), mean_nll=('nll', 'mean'),
    ).round(3).sort_values('n', ascending=False)
    print('\n--- Performance por clase de torneo ---')
    print(by_tournament)

---
## 10. Persistir modelos entrenados

Para Fase 6 (simulación del Mundial 2026).

In [ ]:
import pickle

for name, model in trained_models.items():
    with open(CHECKPOINTS / f'{name}.pkl', 'wb') as f:
        pickle.dump(model, f)

with open(CHECKPOINTS / 'feature_columns.pkl', 'wb') as f:
    pickle.dump(all_cols, f)

print(f'✓ Modelos guardados en {CHECKPOINTS}')
print(f'  - {len(trained_models)} modelos')
print(f'  - feature_columns.pkl con {len(all_cols)} columnas')

---
## 11. Conclusiones de Fase 1

Llenar al final de la ejecución:

- [ ] Mejor modelo por log-loss en TEST: ______
- [ ] Mejor modelo por ECE en TEST (calibración): ______
- [ ] Log-loss del mejor modelo: ______ (uniform = 1.0986)
- [ ] ¿Hay un modelo que destaque claramente o están todos cerca? ______
- [ ] Features más importantes (gradient boosting): ______
- [ ] Régimen donde el modelo pierde más (parejos / mismatch / clase de torneo): ______

## Verificación de comprensión

**(a)** El ELO baseline NO tiene features rolling ni h2h — solo usa la fórmula matemática del ELO. Si XGBoost le gana solo marginalmente al ELO solo, ¿qué dice eso sobre el **valor incremental** de toda la ingeniería de features? Conectalo con el principio de Occam.

**(b)** Para el Mundial 2026, las predicciones que más nos van a importar son partidos de eliminatoria (R16, QF, SF, Final) — exactamente los más raros en training (~32 partidos por Mundial × 2 Mundiales = ~64 partidos). ¿Qué riesgo metodológico introduce esto, y cómo lo mitigaríamos con stratified evaluation en Fase 5?

**(c)** Un modelo A predice [0.51, 0.30, 0.19] y otro modelo B predice [0.99, 0.005, 0.005]. Ambos predicen HOME_WIN. Si el resultado real es HOME_WIN, ambos tienen accuracy=1. Pero el log-loss de A vs B es muy distinto: el de A es ~0.67, el de B es ~0.01. ¿Por qué este comportamiento del log-loss es **deseable** en el contexto de predicción deportiva?